# Cleaning and set up

First let's import everything needed, make sure that notebook is running on the correct virtual environment with all the requirements installed from requirements.txt

In [1]:
import sqlite3 as lite
import pandas as pd
import json

Create a connection to the database, you may change the name if you want a different name for the database or if you want it to be saved in a different path. Note that other files use the same path as well, so you need to update them as well

In [2]:
conn = lite.connect("../database/database.db")
cursor = conn.cursor()

Next, convert data to SQLite database. Downloading DB Browser for SQLite is recommended to easily view the data. Change the function to use the correct file. 


NOTE: will replace any existing tables in the database with the same name!

In [3]:

def convert_excel_to_sql(file: str):
    """ create SQLite database, replaces spaces in sheet names with underscores"""
    all_sheets = pd.read_excel(file, sheet_name=None)
    sheets = all_sheets.keys()
    for sheet_name in sheets:
        sheet = pd.read_excel(file,sheet_name=sheet_name)
        sheet_name = sheet_name.replace(" ", "_")
        query = f"DROP TABLE IF EXISTS {sheet_name};"
        cursor.execute(query)
        conn.commit()
        sheet.to_sql(sheet_name,conn, index=False)

# change this to connect to the correct file
convert_excel_to_sql('../data/xlsx(dont use)/Ridley_et_al_13750_2022_279_MOESM4_ESM2.xlsx')


Next, use pandas to read and edit SQL tables, make sure that the table names match. Also cleans any possible unnecessary spaces from data

In [4]:

def read_sql(table):
    query = f"SELECT * FROM {table}; "
    df = pd.read_sql(query,conn)
    return df

completed = read_sql("Completed_Data_Collection_Tool") #Article level table with different threats combined
seperated = read_sql("Data_Dissagregated") #Table with rows seperated by threat
bib = read_sql("Full_Bibliography") #file containing article titles etc.

completed.columns = [c.strip() for c in completed.columns]
seperated.columns = [c.strip() for c in seperated.columns]
bib.columns = [c.strip() for c in bib.columns]

print("Threats:", seperated.shape, "Unique articles:", seperated["ArticleID"].nunique())


Threats: (23186, 26) Unique articles: 1069


Clean threats and save them to their own table

In [5]:
# Load threat decoding mappings
with open("../data/processed/threat_codes.json", "r", encoding="utf-8") as f:
    threat_data = json.load(f)

threat_codes = threat_data["threat_codes"]
threat_categories = threat_data["threat_categories"]

# Select threat-related columns
threat_df = seperated[
    [
        "ArticleID",
        "Threat",
        "Threat1",
        "Threat_metric",
        "Threat_data",
        "Quantity_threats",
        "threat_precision",
        "threat_database",
    ]
].copy()

# Clean text columns
for col in ["Threat", "Threat1", "Threat_metric", "Threat_data", "threat_precision", "threat_database", "Georef_ind_driver", 'Direct_driver', 'Indirect_driver']:
    if col in threat_df.columns:
        threat_df[col] = threat_df[col].astype(str).str.strip()

# Remove rows where both Threat and Threat1 are missing/empty
threat_df = threat_df[
    ~(
        threat_df["Threat"].replace("nan", "").eq("") &
        threat_df["Threat1"].replace("nan", "").eq("")
    )
].copy()

# Fix known raw-code typos/format issues before decoding
threat_df["Threat"] = threat_df["Threat"].replace({
    "2.5:AgUncpec": "2.5:AgUnspec",
    "4.5.UnspecLine": "4.5:UnspecLine"
})


# Decode full threat code (e.g. 2.3:AgLivestock -> Livestock Farming & Ranching)
threat_df["Threat_decoded"] = threat_df["Threat"].map(threat_codes)

# Decode high-level threat category (e.g. 2 -> Agriculture & Aquaculture)
threat_df["Threat1_decoded"] = threat_df["Threat1"].astype(str).map(threat_categories)

# Save clean processed dataframe

query = """DROP TABLE IF EXISTS Threats_Clean;"""
cursor.execute(query)
conn.commit()
threat_df.to_sql("Threats_Clean", conn, index=False)


print("Saved Threats_Clean")
print(threat_df.head())
print("Columns:", threat_df.columns.tolist())
print("Rows:", len(threat_df))
print("Missing decoded Threat labels:", threat_df["Threat_decoded"].isna().sum())

Saved Threats_Clean
  ArticleID           Threat Threat1 Threat_metric  Threat_data  \
0     25370   8.1:SDAlienInv       8        survey  Qualitative   
1     23013     1.1:DevResid       1    literature  Qualitative   
2     23013  2.3:AgLivestock       2    literature  Qualitative   
3     23013   5.1:UseAnimals       5    literature  Qualitative   
4     23013        6.3:Other       6    literature  Qualitative   

   Quantity_threats threat_precision threat_database  \
0                 1           level2             NaN   
1                 4       cumulative              na   
2                 4       cumulative              na   
3                 4       cumulative              na   
4                 4       cumulative              na   

                      Threat_decoded                       Threat1_decoded  
0  Invasive Non-Native/Alien Species        Invasive & Problematic Species  
1              Housing & Urban Areas  Residential & Commercial Development  
2       L

Merge title column from the bibliography table so a single table can be used for the dashboard. 

You may edit this function or add another if you plan on creating more charts that need other columns from the bibliography table. Remember to rerun the any needed parts from the notebook.

In [6]:
def merge_title(bib_table,completed_table):
    """Change table to be the name of the table containing Title row"""

    # Keep only ArticleID + Title from bibliography
    bib_small = bib_table[["ArticleID", "Title"]].drop_duplicates(subset=["ArticleID"])

    # Merge Title into processed ridley
    out = completed_table.merge(bib_small, on="ArticleID", how="left")

    print("Title added:", "Title" in out.columns)
    print("Missing Title %:", out["Title"].isna().mean())
    print(out[["ArticleID", "Title"]].head(5))

    # Save back to the processed file
    return out


processed = merge_title(bib, completed)

Title added: True
Missing Title %: 0.0
  ArticleID                                              Title
0     25370  Species-specific pattern of crayfish distribut...
1     23013  Spatial variation in leopard (Panthera pardus)...
2     23018  Dynamic ensemble models to predict distributio...
3     23019  Mapping perceptions of species' threats and po...
4     23020  Social media as a novel source of data on the ...


Clean Authors row

In [ ]:
processed["Authors"] = processed["Authors"].replace({"_etal": " et al."}, regex=True)

0             Rímalová et al.
1                Abade et al.
2              Abrahms et al.
3                Abram et al.
4                Abreo et al.
                ...          
1064           Ziegler et al.
1065            Zimkus et al.
1066    Zingraff-Hamed et al.
1067             Zomer et al.
1068           Žydelis et al.
Name: Authors, Length: 1069, dtype: str

Make sure all missing values are in the same format

In [8]:


# Standardize missing values (convert empty strings to NA)
na_like = ["", " ", "NA", "N/A", "na", "n/a", "null", "None", "none", "-", "--"]
processed = processed.replace(na_like, pd.NA)

print("Dataframe shape:", processed.shape)
print("Unique articles (ArticleID):", processed["ArticleID"].nunique())


Dataframe shape: (1069, 28)
Unique articles (ArticleID): 1069


Clean Driver columns

In [9]:
def clean_georef(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value == "1":
        return "Yes"
    elif value == "0":
        return "No"
    elif value == "na":
        return "Unknown"
    elif ";" in value:
        return "Mixed"
    else:
        return "Unknown"

def clean_driver_text(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value == "na":
        return "Unknown"

    value = value.replace("_", " ")
    return value

processed["Georef_ind_driver"] = completed["Georef_ind_driver"].apply(clean_georef)
processed["Direct_driver"] = completed["Direct_driver"].apply(clean_driver_text)
processed["Indirect_driver"] = completed["Indirect_driver"].apply(clean_driver_text)
print("\nPreview of cleaned columns:")
print(processed[[
    "Georef_ind_driver",
    "Direct_driver",
    "Indirect_driver",
]].head(10))


Preview of cleaned columns:
  Georef_ind_driver                                      Direct_driver  \
0               Yes                                     climate change   
1                No     invasive and alien species;land/sea use change   
2                No                                land/sea use change   
3             Mixed                 climate change;direct exploitation   
4               Yes                 climate change;land/sea use change   
5             Mixed     direct exploitation;invasive and alien species   
6             Mixed                      land/sea use change;pollution   
7               Yes                      direct exploitation;pollution   
8               Yes               invasive and alien species;pollution   
9               Yes  climate change;direct exploitation;invasive an...   

                                     Indirect_driver  
0                             conflits and epidemics  
1                 conflits and epidemics;demog

# check missingness

In [10]:
cols_check = [c for c in ["Authors","Year","Study_design","Ecoregion","Continent_Ocean","Country_EEZ","Georef_ind_driver", "Direct_driver", "Indirect_driver"] if c in processed.columns]

for c in cols_check:
    print(c, round(processed[c].isna().mean()*100, 1), "% missing")


Authors 0.0 % missing
Year 0.0 % missing
Study_design 0.0 % missing
Ecoregion 0.0 % missing
Continent_Ocean 9.2 % missing
Country_EEZ 13.1 % missing
Georef_ind_driver 0.0 % missing
Direct_driver 0.0 % missing
Indirect_driver 0.0 % missing


### Missing values (article-level table)

Key metadata fields are mostly complete:
- Authors, Year, Study_design, Ecoregion: **0% missing**

Some location-related fields are partially missing:
- Continent_Ocean: **9.2% missing**
- country_eez: **13.1% missing**

We will keep these as missing (NA) and handle them in the dashboard by showing them as **“Unknown/Not specified”** rather than imputing.


### Dashboard-ready version (handling missing region fields)

For dashboard filters, we replace missing values in `Continent_Ocean` and `country_eez` with `"Unknown"`.
This avoids empty categories during filtering while keeping the original data unchanged.


In [11]:

processed["Continent_Ocean"] = processed["Continent_Ocean"].fillna("Unknown")
processed["Country_EEZ"] = processed["Country_EEZ"].fillna("Unknown")

cursor.execute("""DROP TABLE IF EXISTS processed """)
conn.commit()
processed.to_sql("processed",conn,index=False)
print("Saved: processed")


Saved: processed


### Top 10 threats


In [12]:
tmp = processed[["ArticleID", "Threat"]].dropna().copy()
tmp["Threat"] = tmp["Threat"].astype(str).str.strip()

top10 = (tmp.drop_duplicates()
           .groupby("Threat")["ArticleID"]
           .nunique()
           .sort_values(ascending=False)
           .head(10))

print(top10)


Threat
8.1:SDAlienInv     125
5.4:UseAquatic     107
5.1:UseAnimals      64
4.1:RoadRail        55
7.2:WaterSystem     21
11.1:CC_Habitat     18
2.5:AgUncpec        18
4.3:ShipLane        17
3.3:Renew           17
11.3:CC_Temp        14
Name: ArticleID, dtype: int64


Delete any SQL tables that are unnecessary. You may edit this and run the file again if needed

In [13]:
cursor.execute("""DROP TABLE IF EXISTS README;""")
cursor.execute("""DROP TABLE IF EXISTS Extended_Definitions; """)
cursor.execute("""DROP TABLE IF EXISTS Typology; """)
cursor.execute("""DROP TABLE IF EXISTS Search; """)
conn.commit()

Close SQL connection

In [14]:
conn.close()